In [ ]:
!pip install -q transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 evaluate==0.4.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 55.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which

In [ ]:
import pandas as pd
import numpy as np

import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

import evaluate

print("Torch:", torch.__version__)
print("GPU:", torch.cuda.is_available())

Torch: 2.11.0+cu128
GPU: True


In [ ]:
df = pd.read_csv("/content/archive.zip")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
# Encode labels

df["label"] = df["sentiment"].map({
    "negative":0,
    "positive":1
})


train_df, temp_df = train_test_split(
    df[["review","label"]],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)


print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 40000
Validation: 5000
Test: 5000


In [ ]:
train_dataset = Dataset.from_pandas(train_df)

val_dataset = Dataset.from_pandas(val_df)

test_dataset = Dataset.from_pandas(test_df)

In [ ]:
MODEL_NAME = "distilbert-base-uncased"


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


def tokenize(batch):

    return tokenizer(
        batch["review"],
        truncation=True,
        max_length=256
    )


train_dataset = train_dataset.map(
    tokenize,
    batched=True
)


val_dataset = val_dataset.map(
    tokenize,
    batched=True
)


test_dataset = test_dataset.map(
    tokenize,
    batched=True
)


train_dataset = train_dataset.remove_columns(["review"])
val_dataset = val_dataset.remove_columns(["review"])
test_dataset = test_dataset.remove_columns(["review"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

In [ ]:
training_args = TrainingArguments(

    output_dir="./distilbert_imdb",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",

    fp16=True,

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    data_collator=DataCollatorWithPadding(
        tokenizer
    ),

    compute_metrics=compute_metrics
)

/tmp/ipykernel_1093/2882542737.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.237800,0.231490,0.914800
2,0.167600,0.247480,0.913800
3,0.102600,0.313686,0.918800


TrainOutput(global_step=7500, training_loss=0.18099132029215495, metrics={'train_runtime': 948.2077, 'train_samples_per_second': 126.555, 'train_steps_per_second': 7.91, 'total_flos': 7948043919360000.0, 'train_loss': 0.18099132029215495, 'epoch': 3.0})

In [ ]:
results = trainer.predict(test_dataset)

print(results.metrics)

{'test_loss': 0.3172384202480316, 'test_accuracy': 0.9204, 'test_runtime': 10.1388, 'test_samples_per_second': 493.154, 'test_steps_per_second': 30.871}


In [ ]:
predictions = np.argmax(
    results.predictions,
    axis=1
)


print(
    classification_report(
        test_df["label"],
        predictions
    )
)

              precision    recall  f1-score   support

           0       0.92      0.92      0.92      2500
           1       0.92      0.92      0.92      2500

    accuracy                           0.92      5000
   macro avg       0.92      0.92      0.92      5000
weighted avg       0.92      0.92      0.92      5000



In [ ]:
# ==========================================
# Save DistilBERT Model to Google Drive
# ==========================================

save_path = "/content/drive/MyDrive/IMDB_DistilBERT_Model"

# Save model
trainer.save_model(save_path)

# Save tokenizer
tokenizer.save_pretrained(save_path)

print("Model and tokenizer saved successfully ")

Model and tokenizer saved successfully 


In [ ]:
# ==========================================
# DistilBERT - New Review Prediction
# ==========================================

import torch
import numpy as np

def predict_sentiment(text):
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    # Use same device as model
    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Prediction
    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    # Convert logits to probabilities
    probabilities = torch.softmax(outputs.logits, dim=-1)

    prediction = torch.argmax(probabilities, dim=-1).item()

    confidence = probabilities[0][prediction].item()

    sentiment = "Positive 😊" if prediction == 1 else "Negative 😞"

    return sentiment, confidence


# ==========================================
# Test Reviews
# ==========================================

test_reviews = [
    "The movie was absolutely fantastic. The story was emotional and the acting was incredible.",

    "I regret spending my evening watching this movie. Nothing about it was interesting.",

    "The visuals were beautiful, but the story was confusing and the characters felt empty.",

    "This film surprised me in the best possible way. I would happily watch it again.",

    "The movie started well, but the ending was disappointing and ruined the whole experience.",

    "I did not expect much from this film, but it turned out to be surprisingly entertaining.",

    "The acting was terrible and the dialogue felt unnatural. I couldn't wait for it to end."
]


for i, review in enumerate(test_reviews, 1):

    sentiment, confidence = predict_sentiment(review)

    print("=" * 80)
    print(f"Review {i}:")
    print(review)
    print(f"\nPrediction : {sentiment}")
    print(f"Confidence : {confidence:.2%}")

Review 1:
The movie was absolutely fantastic. The story was emotional and the acting was incredible.

Prediction : Positive 😊
Confidence : 99.63%
Review 2:
I regret spending my evening watching this movie. Nothing about it was interesting.

Prediction : Negative 😞
Confidence : 99.65%
Review 3:
The visuals were beautiful, but the story was confusing and the characters felt empty.

Prediction : Negative 😞
Confidence : 99.63%
Review 4:
This film surprised me in the best possible way. I would happily watch it again.

Prediction : Positive 😊
Confidence : 99.66%
Review 5:
The movie started well, but the ending was disappointing and ruined the whole experience.

Prediction : Negative 😞
Confidence : 99.66%
Review 6:
I did not expect much from this film, but it turned out to be surprisingly entertaining.

Prediction : Positive 😊
Confidence : 99.50%
Review 7:
The acting was terrible and the dialogue felt unnatural. I couldn't wait for it to end.

Prediction : Negative 😞
Confidence : 99.83%


In [ ]:
# ==========================================
# Professional Model Saving
# ==========================================

import os
import json
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Main save directory
SAVE_DIR = "/content/drive/MyDrive/IMDB_Sentiment_DistilBERT"

os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save model
trainer.save_model(SAVE_DIR)

# 2. Save tokenizer
tokenizer.save_pretrained(SAVE_DIR)

# 3. Save model information
model_info = {
    "model_name": "DistilBERT",
    "base_model": "distilbert-base-uncased",
    "task": "IMDB Sentiment Classification",
    "num_labels": 2,
    "labels": {
        "0": "Negative",
        "1": "Positive"
    },
    "max_length": 256,
    "test_accuracy": 0.9188
}

with open(
    os.path.join(SAVE_DIR, "model_info.json"),
    "w"
) as f:
    json.dump(model_info, f, indent=4)

print("Model saved successfully!")
print(f" Location: {SAVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved successfully!
 Location: /content/drive/MyDrive/IMDB_Sentiment_DistilBERT


In [ ]:
!pip install -q gradio

In [ ]:
import torch
import gradio as gr

from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSequenceClassification

drive.mount("/content/drive")

MODEL_PATH = "/content/drive/MyDrive/IMDB_Sentiment_DistilBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

print("Model loaded successfully ✅")
print("Device:", device)

Mounted at /content/drive


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully ✅
Device: cuda


In [ ]:
def predict_sentiment(text):

    if not text or not text.strip():
        return {
            "No Review": 1.0
        }

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    negative_prob = probabilities[0].item()
    positive_prob = probabilities[1].item()

    return {
        "Positive 😊": positive_prob,
        "Negative 😞": negative_prob
    }

In [ ]:
interface = gr.Interface(
    fn=predict_sentiment,

    inputs=gr.Textbox(
        lines=6,
        placeholder="Write a movie review here...",
        label="Movie Review"
    ),

    outputs=gr.Label(
        num_top_classes=2,
        label="Sentiment Prediction"
    ),

    title="🎬 IMDB Movie Sentiment Analyzer",

    description=(
        "Enter a movie review and the DistilBERT model "
        "will predict whether the sentiment is Positive or Negative."
    ),

    examples=[
        ["This movie was absolutely amazing. I loved every minute of it."],
        ["The story was boring and the acting was terrible."],
        ["The movie started well but the ending was disappointing."]
    ]
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a7bd98a2e1666578c1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
